# 04 Gold

Para la `fecha` del job se reconstruye tres tablas a partir de silver (`fecha_dia`). `replaceWhere`
borra solo ese día y lo vuelve a escribir, así repetir la tarea no duplica.

- **Trayectorias:** una fila por buque, con la distancia del día (haversine, radio 6371 km,
  la misma fórmula de la Parte 1) entre posiciones consecutivas.
- **Presencia en puerto:** el buque tiene al menos una posición en la celda H3 resolución 7
  del puerto o en el anillo vecino (k = 1). La Parte 1 usó resolución 8 exacta y no cruzó
  con ningún puerto, esta vez la celda es más amplia.
- **Tráfico:** posiciones y buques por celda H3 resolución 8 y hora UTC.


In [ ]:
%run ./_config


## Trayectorias


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

iso = fecha_dia.isoformat()
dia = (
    spark.table(SILVER)
    .where(F.col("fecha_dia") == F.lit(iso).cast("date"))
    .withColumn("fecha", F.col("fecha_dia"))
)

def trayectorias(posiciones):
    orden = Window.partitionBy("MMSI").orderBy("BaseDateTime")
    orden_fin = Window.partitionBy("MMSI").orderBy(F.desc("BaseDateTime"))
    lat1 = F.radians(F.lag("LAT").over(orden))
    lon1 = F.radians(F.lag("LON").over(orden))
    lat2 = F.radians("LAT")
    lon2 = F.radians("LON")
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = F.sin(dlat / 2) ** 2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2
    tramo = F.when(
        F.lag("LAT").over(orden).isNotNull(),
        2 * F.lit(6371.0) * F.asin(F.sqrt(a)),
    ).otherwise(F.lit(0.0))
    marcado = (
        posiciones.withColumn("tramo_km", tramo)
        .withColumn("lat_inicio", F.first("LAT").over(orden))
        .withColumn("lon_inicio", F.first("LON").over(orden))
        .withColumn("lat_fin", F.first("LAT").over(orden_fin))
        .withColumn("lon_fin", F.first("LON").over(orden_fin))
        .withColumn("inicio", F.first("BaseDateTime").over(orden))
        .withColumn("fin", F.first("BaseDateTime").over(orden_fin))
    )
    return marcado.groupBy("fecha", "MMSI").agg(
        F.count("*").alias("n_posiciones"),
        F.max("inicio").alias("inicio"),
        F.max("fin").alias("fin"),
        F.max("lat_inicio").alias("lat_inicio"),
        F.max("lon_inicio").alias("lon_inicio"),
        F.max("lat_fin").alias("lat_fin"),
        F.max("lon_fin").alias("lon_fin"),
        F.sum("tramo_km").alias("distancia_km"),
        F.avg("SOG").alias("sog_media"),
    )


def publicar(df, tabla: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"fecha = '{iso}'")
        .saveAsTable(tabla)
    )
    print("Publicado", tabla)


publicar(trayectorias(dia), GOLD_TRAY)
display(spark.table(GOLD_TRAY).where(F.col("fecha") == F.lit(iso).cast("date")).limit(10))


## Presencia en puerto


In [ ]:
puertos = (
    spark.table(PUERTOS)
    .where(F.col("latitud").isNotNull() & F.col("longitud").isNotNull())
    .withColumn(
        "h3_res7",
        F.explode(F.expr(
            "transform(h3_kring(h3_longlatash3(longitud, latitud, 7), 1), c -> h3_h3tostring(c))"
        )),
    )
    .select("nombre", "pais", "h3_res7")
)
posiciones = dia.withColumn("h3_res7", F.expr("h3_longlatash3string(LON, LAT, 7)"))
presencia = (
    posiciones.join(puertos, "h3_res7", "inner")
    .groupBy("fecha", "MMSI", "nombre", "pais")
    .agg(
        F.count("*").alias("n_posiciones"),
        F.min("BaseDateTime").alias("inicio"),
        F.max("BaseDateTime").alias("fin"),
    )
    .withColumnRenamed("nombre", "puerto")
)
publicar(presencia, GOLD_PUERTO)
display(spark.table(GOLD_PUERTO).where(F.col("fecha") == F.lit(iso).cast("date")).limit(10))


## Tráfico por celda y hora


In [ ]:
trafico = (
    dia.withColumn("hora", F.hour("BaseDateTime"))
    .withColumn("h3_cell", F.expr("h3_longlatash3string(LON, LAT, 8)"))
    .groupBy("fecha", "hora", "h3_cell")
    .agg(
        F.count("*").alias("n_posiciones"),
        F.countDistinct("MMSI").alias("n_buques"),
    )
)
publicar(trafico, GOLD_TRAFICO)
display(spark.table(GOLD_TRAFICO).where(F.col("fecha") == F.lit(iso).cast("date")).orderBy(F.desc("n_posiciones")).limit(10))
